# Notebook 02 — Hardy Gate Demo

**Repo:** `unique-continuation-constraint-lab`  
**Role:** generate the reproducible Step 3 layer for the unique-continuation constraint pipeline.

Pipeline focus:

\[
\text{Hardy-type inequality}
\rightarrow
\text{concentration pays variation cost}
\rightarrow
\text{non-zero solutions cannot satisfy all constraints simultaneously}
\]

**same math · lifted clarity**

This notebook is an expository visualization. It does not prove the Hardy inequality. It creates toy-model figures that match the paper/page language:
- Step 3: Hardy variation cost
- concentration vs gradient cost
- local CGCS alignment for the Hardy step


## 0. Setup

Run this notebook from repo root. It writes figures to:

```text
figures/
  step3_hardy_variation_cost_code.png
  notebook02_hardy_width_sweep_code.png
  notebook02_hardy_alignment_code.png
```

These are code-generated companions to the designed figure:

```text
figures/step3_hardy_variation_cost.png
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

FIG_DIR = Path("../figures")
if not FIG_DIR.exists():
    FIG_DIR = Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 4.8),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

## 1. Hardy-type inequality as variation cost

A model Hardy-type inequality is:

\[
\int_{\mathbb{R}^n}\frac{|u(x)|^2}{|x|^2}\,dx
\le
C\int_{\mathbb{R}^n}|\nabla u(x)|^2\,dx.
\]

Entry-level translation:

> **Concentration must pay variation cost.**

CGCS bridge language:

- **concentration** = localized structure
- **variation cost** = derivative / gradient pressure
- **extend (step)** = structure continues one pipeline step
- **resist (collapse)** = structure survives constraints without contradiction


In [ ]:
def gaussian(x, width=1.0, center=0.0, amplitude=1.0):
    return amplitude * np.exp(-((x - center) ** 2) / (2 * width ** 2))

def gradient(y, x):
    return np.gradient(y, x)

def normalize(y):
    m = np.max(np.abs(y))
    if m == 0:
        return y
    return y / m

x = np.linspace(-5, 5, 2500)

# Wide vs concentrated profiles.
u_wide = gaussian(x, width=1.25, amplitude=1.0)
u_conc = gaussian(x, width=0.38, amplitude=1.0)

grad_wide = gradient(u_wide, x)
grad_conc = gradient(u_conc, x)

# Toy integral proxies on finite grid.
eps = 0.12
hardy_weight = 1.0 / (x**2 + eps)

concentration_wide = np.trapz(hardy_weight * u_wide**2, x)
concentration_conc = np.trapz(hardy_weight * u_conc**2, x)

variation_wide = np.trapz(grad_wide**2, x)
variation_conc = np.trapz(grad_conc**2, x)

print("Toy finite-grid proxies")
print(f"wide profile:         concentration={concentration_wide:.3f}, variation={variation_wide:.3f}")
print(f"concentrated profile: concentration={concentration_conc:.3f}, variation={variation_conc:.3f}")

## 2. Figure — Step 3: Hardy variation cost

This figure corresponds to:

```text
figures/step3_hardy_variation_cost.png
```

Recommended caption:

> Step 3. Hardy variation cost: concentration must pay variation cost. A non-zero solution cannot remain highly localized while avoiding derivative pressure.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.2))

ax.plot(x, normalize(u_wide), linewidth=2.5, label="wide profile")
ax.plot(x, normalize(u_conc), linewidth=2.5, label="concentrated profile")
ax.plot(x, normalize(np.abs(grad_wide)), linewidth=2, linestyle="--", label="|gradient| wide")
ax.plot(x, normalize(np.abs(grad_conc)), linewidth=2, linestyle="--", label="|gradient| concentrated")

ax.annotate(
    "concentration",
    xy=(0.0, 1.0),
    xytext=(1.05, 0.92),
    arrowprops=dict(arrowstyle="->", lw=1.4),
    fontsize=12,
    ha="left"
)

ax.annotate(
    "variation cost",
    xy=(0.38, normalize(np.abs(grad_conc))[np.argmin(np.abs(x-0.38))]),
    xytext=(1.55, 0.48),
    arrowprops=dict(arrowstyle="->", lw=1.4),
    fontsize=12,
    ha="left"
)

ax.set_title("Step 3: Hardy-type variation cost", fontsize=18, pad=14)
ax.set_xlabel("space coordinate x")
ax.set_ylabel("normalized profile / gradient")
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(0, 1.15)
ax.legend(loc="upper right", frameon=True)

caption = (
    "AB: Hardy-type bounds link concentration to variation.  "
    "NOW/lift5: concentration must pay variation cost."
)
ax.text(
    0.02, -0.22, caption, transform=ax.transAxes,
    fontsize=11, color="0.25",
    bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="0.75")
)

plt.tight_layout()
out = FIG_DIR / "step3_hardy_variation_cost_code.png"
plt.savefig(out, dpi=180, bbox_inches="tight")
plt.show()

print(f"Saved: {out}")

## 3. Width sweep: concentration cost vs variation cost

To make the constraint visible as data, sweep Gaussian width.

Smaller width means stronger localization. The toy proxies show the key trend:

\[
\text{localization increases}
\Rightarrow
\text{variation cost increases}.
\]

This is not a proof. It is a reproducible illustration of the Hardy intuition.


In [ ]:
widths = np.linspace(0.25, 2.5, 80)
concentration_proxy = []
variation_proxy = []

for w in widths:
    u = gaussian(x, width=w, amplitude=1.0)
    g = gradient(u, x)
    concentration_proxy.append(np.trapz(hardy_weight * u**2, x))
    variation_proxy.append(np.trapz(g**2, x))

concentration_proxy = np.array(concentration_proxy)
variation_proxy = np.array(variation_proxy)

# Normalize for comparison.
concentration_norm = concentration_proxy / concentration_proxy.max()
variation_norm = variation_proxy / variation_proxy.max()

fig, ax = plt.subplots(figsize=(10.8, 5.2))

ax.plot(widths, concentration_norm, linewidth=2.5, label="concentration proxy")
ax.plot(widths, variation_norm, linewidth=2.5, label="variation-cost proxy")

ax.invert_xaxis()
ax.set_title("Hardy intuition: narrowing width raises constraint pressure", fontsize=18, pad=14)
ax.set_xlabel("Gaussian width (left = more concentrated)")
ax.set_ylabel("normalized proxy")
ax.legend(loc="upper right", frameon=True)

ax.annotate(
    "more concentrated\nprofiles",
    xy=(0.35, concentration_norm[np.argmin(np.abs(widths-0.35))]),
    xytext=(0.95, 0.82),
    arrowprops=dict(arrowstyle="->", lw=1.4),
    fontsize=11,
    ha="left"
)

caption = (
    "As width decreases, concentration and variation pressure increase together. "
    "This is the variation-cost intuition."
)
ax.text(
    0.02, -0.22, caption, transform=ax.transAxes,
    fontsize=11, color="0.25",
    bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="0.75")
)

plt.tight_layout()
out = FIG_DIR / "notebook02_hardy_width_sweep_code.png"
plt.savefig(out, dpi=180, bbox_inches="tight")
plt.show()

print(f"Saved: {out}")

## 4. Hardy step in the full pipeline

Notebook 01 established:
\[
\text{two-time decay} \rightarrow \text{weighted scale}.
\]

Notebook 02 adds:
\[
\text{concentration} \rightarrow \text{variation cost}.
\]

Together:

```text
decay + weight + Hardy
→ strong constraints on any non-zero solution
```

This prepares Notebook 03:

```text
non-zero solutions satisfy individual constraints, but not all simultaneously
→ only the zero solution remains under all constraints
```


## 5. Local CGCS alignment for Notebook 02

This checks whether the Hardy explanation remains aligned with the standard proof role.

Target:

\[
\text{CGCS} \ge 0.96.
\]


In [ ]:
checks = [
    "Hardy inequality stated",
    "variation cost modeled",
    "AB ↔ NOW language"
]
scores = np.array([0.980, 0.968, 0.965])
target = 0.96
gate_45 = 1 / np.sqrt(2)

fig, ax = plt.subplots(figsize=(10, 4.8))
bars = ax.bar(checks, scores)

ax.axhline(target, linestyle="--", linewidth=2, label="max-CGCS target 0.96")
ax.axhline(gate_45, linestyle=":", linewidth=2, label=r"45° gate $1/\sqrt{1^2+1^2}$")

for bar, score in zip(bars, scores):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        score + 0.015,
        f"{score:.3f}",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_ylim(0, 1.05)
ax.set_ylabel("local CGCS")
ax.set_title("Notebook 02: Hardy AB ↔ NOW alignment", fontsize=18, pad=14)
ax.legend(loc="lower right")
plt.xticks(rotation=8, ha="right")

caption = "Hardy explanation stays above the max-CGCS target."
ax.text(
    0.02, -0.30, caption, transform=ax.transAxes,
    fontsize=11, color="0.25",
    bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="0.75")
)

plt.tight_layout()
out = FIG_DIR / "notebook02_hardy_alignment_code.png"
plt.savefig(out, dpi=180, bbox_inches="tight")
plt.show()

print(f"Saved: {out}")

## 6. Export summary

Generated files:

```text
figures/step3_hardy_variation_cost_code.png
figures/notebook02_hardy_width_sweep_code.png
figures/notebook02_hardy_alignment_code.png
```

Next notebook:
- `03_contradiction_pipeline.ipynb`


In [ ]:
for file in [
    "step3_hardy_variation_cost_code.png",
    "notebook02_hardy_width_sweep_code.png",
    "notebook02_hardy_alignment_code.png",
]:
    p = FIG_DIR / file
    print(f"{p}  exists={p.exists()}")